# Graph RAG
SoTA RAG는 최신 변형이 많아 복잡하고, Agentic RAG는 LangGraph 같은 에이전트 워크플로우가 필요해 초보자에 부담스럽습니다. 반면 Graph RAG는 지식 그래프(노드+엣지)를 활용해 관계 기반 검색을 하는데, 간단한 예제로 빠르게 구현 가능



```
# 코드로 형식 지정됨
```

### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
%cd /content/drive/MyDrive/aiffel/3.rag

/content/drive/MyDrive/aiffel/3.rag


In [9]:
!pwd

/content/drive/MyDrive/aiffel/3.rag


In [10]:
from dotenv import load_dotenv
import os

load_dotenv("/content/drive/MyDrive/aiffel/env_keys/.env")

print("OPENAI_API_KEY loaded:", os.getenv("OPENAI_API_KEY") is not None)
print("NEO4J_USERNAME loaded:", os.getenv("NEO4J_USERNAME") is not None)
print("NEO4J_PASSWORD loaded:", os.getenv("NEO4J_PASSWORD") is not None)
print("NEO4J_DATABASE loaded:", os.getenv("NEO4J_DATABASE") is not None)


NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")


OPENAI_API_KEY loaded: True
NEO4J_USERNAME loaded: True
NEO4J_PASSWORD loaded: True
NEO4J_DATABASE loaded: True


In [ ]:
# YOUR_API_KEY = ''

### 설치:

In [1]:
!pip install --upgrade langchain-core langchain-community langchain-neo4j
!pip install --upgrade langchain langgraph openai langchain-openai langchain-experimental

  Using cached langgraph-1.0.10-py3-none-any.whl.metadata (7.4 kB)
Using cached langgraph-1.0.10-py3-none-any.whl (160 kB)
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.2
    Uninstalling langgraph-1.1.2:
      Successfully uninstalled langgraph-1.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 1.2.12 requires langgraph<1.2.0,>=1.1.1, but you have langgraph 1.0.10 which is incompatible.
  Using cached langgraph-1.1.2-py3-none-any.whl.metadata (7.4 kB)
Using cached langgraph-1.1.2-py3-none-any.whl (167 kB)
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.0.10
    Uninstalling langgraph-1.0.10:
      Successfully uninstalled langgraph-1.0.10
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

In [2]:
# !pip install langchain langchain-community langgraph neo4j openai

In [3]:
# !pip install langchain-openai

In [4]:
# !pip install langchain-experimental

In [5]:
# !pip install langchain-community langchain-classic

### import

In [3]:
# from langchain_community.graphs import Neo4jGraph  # 또는 Memgraph
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI
# from langchain_experimental.graph_transformers import LLMGraphTransformer
# from langchain_core.documents import Document
# from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain

# from langgraph.graph import NetworkGraph  # 새 경로!
# from langchain_community.chains import GraphCypherQAChain
# from langchain_openai import ChatOpenAI
# import os

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

### 클라우드 Neo4j(Aura) + Colab
윈도우는 그냥 브라우저용, Neo4j는 클라우드에 두고 Colab에서 붙는 방식

- Aura는 Neo4j를 클라우드에서 완전 관리형 서비스(DBaaS)로 제공하는 플랫폼입니다.
<details>
<summary>details</summary>
1. Aura가 하는 핵심 역할
- Neo4j 그래프 데이터베이스를 “설치·운영” 걱정 없이 클라우드에서 바로 쓰게 해주는 서비스입니다.

- 서버 인프라, 백업, 패치, 업그레이드, 모니터링 같은 운영을 Neo4j 측이 대신 맡고, 사용자는 Cypher로 쿼리만 날리면 됩니다.

2. 두 가지 주요 서비스 (AuraDB / AuraDS)
- AuraDB: 트랜잭션 그래프 DB 서비스로, 애플리케이션에서 데이터 저장·조회·추천·경로 탐색 같은 그래프 쿼리를 수행할 때 사용합니다.

- AuraDS: Graph Data Science(GDS)용으로, PageRank, 커뮤니티 탐지, 경로 최적화 등 그래프 알고리즘·예측 모델을 돌리기 위한 데이터 사이언스 워크로드에 맞춰진 서비스입니다.
</details>

- Neo4j Aura 사이트에서 무료 인스턴스 생성
- Aura 콘솔에서 제공하는:
  - Bolt URL
  - username (neo4j)
  - password (자동 생성)
를 복사.


#### Neo4j를 python에서 활용 (그래프 DB 연결)

In [4]:
# !pip install neo4j

In [5]:
#!pip install -U langchain-neo4j

In [11]:
from langchain_neo4j import Neo4jGraph

graph = Neo4jGraph(
    url="neo4j+s://8b4976cd.databases.neo4j.io",
    username=NEO4J_USERNAME,   # "neo4j",
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,  # "neo4j",
)

In [12]:
# # 1. langchain-community 최신 버전 확인/업데이트
# !pip install --upgrade langchain-community langchain-openai langchain-experimental neo4j

#### LLM 설정

In [13]:
from langchain_community.graphs import Neo4jGraph
from langchain_community.chains.graph_qa.cypher import GraphCypherQAChain # Changed import path
from langchain_openai import ChatOpenAI

# OpenAI
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# test
response = llm.invoke("안녕하세요!")
print(response.content)

안녕하세요! 어떻게 도와드릴까요?


In [14]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

transformer = LLMGraphTransformer(llm=llm)

### 샘플 텍스트 → 그래프 변환

In [28]:
from langchain_core.documents import Document

docs = [Document(page_content="현재 Apple의 CEO는 Tim Cook. Steve Jobs는 전 CEO. Tim Cook은 Steve Jobs의 후임자. iPhone은 Apple 제품.")]
graph_docs = transformer.convert_to_graph_documents(docs)
graph.add_graph_documents(graph_docs)

### # 5. Cypher QA 체인으로 쿼리

In [36]:
# 1) Apple 관련 모든 관계 제거
graph.query("""
MATCH (c:Company {id: 'Apple'})-[r]->()
DELETE r;
""")
graph.refresh_schema()

In [37]:
# // 2) SUCCESSOR 관계 전부 제거 (Steve Jobs / Tim Cook 관련)
graph.query("""
MATCH (p1:Person)-[r:SUCCESSOR]->(p2:Person)
WHERE p1.id IN ['Steve Jobs', 'Tim Cook'] OR p2.id IN ['Steve Jobs', 'Tim Cook']
DELETE r;
""")
graph.refresh_schema()

In [38]:
graph.query("""
MATCH (n) RETURN n;
""")
graph.refresh_schema()

In [41]:
graph.query("""
MERGE (apple:Company {id: 'Apple'})
MERGE (jobs:Person {id: 'Steve Jobs'})
MERGE (cook:Person {id: 'Tim Cook'})
MERGE (apple)-[:CURRENT_CEO]->(cook)
MERGE (jobs)-[:SUCCESSOR]->(cook)
""")
graph.refresh_schema()

In [44]:
# 1) Apple → CURRENT_CEO
res1 = graph.query("""
MATCH (c:Company {id: 'Apple'})-[:CURRENT_CEO]->(p:Person)
RETURN c.id AS Company, p.id AS Current_CEO
""")
print(res1)

# 2) Steve Jobs → SUCCESSOR
res2 = graph.query("""
MATCH (jobs:Person {id: 'Steve Jobs'})-[:SUCCESSOR]->(next:Person)
RETURN jobs.id AS Former_CEO, next.id AS Successor
""")
print(res2)

graph.refresh_schema()

[{'Company': 'Apple', 'Current_CEO': 'Tim Cook'}]
[{'Former_CEO': 'Steve Jobs', 'Successor': 'Tim Cook'}]


In [45]:
from langchain_neo4j import GraphCypherQAChain

chain = GraphCypherQAChain.from_llm(graph=graph, llm=llm, verbose=True, allow_dangerous_requests=True)
result = chain.invoke({"query": "Apple CEO는 누구? 후임자 관계는?"})
print(result['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {id: 'Apple'})-[:CURRENT_CEO]->(p:Person)-[:SUCCESSOR]->(s:Person) RETURN p, s
Full Context:
[]

> Finished chain.
저는 그에 대한 정보를 알지 못합니다.


In [46]:
result = chain.invoke({"query": "Apple의 현재 CEO와 Steve Jobs의 후임자를 알려줘"})
print(result['result'])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {id: 'Apple'})-[:CURRENT_CEO]->(currentCeo:Person), 
      (currentCeo)-[:SUCCESSOR]->(successor:Person) 
RETURN currentCeo, successor
Full Context:
[]

> Finished chain.
현재 Apple의 CEO는 팀 쿡(Tim Cook)이며, 그는 스티브 잡스(Steve Jobs)의 후임자입니다.


In [47]:
result = chain.invoke({"query": "Steve Jobs의 후임자는 누구야?"})
print(result['result'])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Person {id: 'Steve Jobs'})-[:SUCCEEDED_BY]->(successor:Person) RETURN successor.id
Full Context:
[{'successor.id': 'Tim Cook'}]

> Finished chain.
Steve Jobs의 후임자는 Tim Cook입니다.


In [49]:
result = chain.invoke({"query": "Apple CEO는 누구야?"})
print(result['result'])




> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {id: 'Apple'})-[:CURRENT_CEO]->(p:Person) RETURN p.id
Full Context:
[{'p.id': 'Tim Cook'}]

> Finished chain.
Apple CEO는 Tim Cook입니다.


In [50]:
result = chain.invoke({"query": "전임 Apple CEO는 누구야?"})



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:Company {id: 'Apple'})-[:CURRENT_CEO]->(p:Person)-[:SUCCEEDED_BY]->(previousCEO:Person) RETURN previousCEO.id
Full Context:
[]

> Finished chain.


간단한 테스트 종료

### 의존성 모듈 설치

In [51]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [52]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

In [53]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': 'Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [54]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [55]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

### 1. PDF 로드 + 청크 만들기

In [58]:
!pip install -q langchain-text-splitters

In [60]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150,)
docs = text_splitter.split_documents(pages)
print(len(docs), docs[0].page_content[:200])

360 DEMIAN 
• 
Downloaded from https://www.holybooks.com


### 2. Neo4j Aura를 **벡터 스토어(Neo4jVector)**로 사용
- Demian 청크들을 Neo4j Aura에 노드로 저장
- 각 청크에 임베딩을 계산해서 벡터 인덱스까지 Aura 안에 생성
- 따로 Chromadb는 필요 없습니다.

In [61]:
from langchain_neo4j import Neo4jVector
from langchain_openai import OpenAIEmbeddings

url = "neo4j+s://8b4976cd.databases.neo4j.io"
username = NEO4J_USERNAME
password = NEO4J_PASSWORD

embeddings = OpenAIEmbeddings()

vector_store = Neo4jVector.from_documents(
    documents=docs,
    embedding=embeddings,
    url=url,
    username=username,
    password=password,
    index_name="demian_index",      # 원하는 이름
)


### 3. RAG 체인 만들기 (Aura만 사용)

In [63]:
!pip show langchain

Name: langchain
Version: 1.2.12
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 


In [65]:
!pip install langchain-classic

In [66]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

retriever = vector_store.as_retriever(search_kwargs={"k": 4})

prompt = ChatPromptTemplate.from_template(
    """당신은 친절한 AI 어시스턴트입니다.
다음 컨텍스트만 사용해서 질문에 답하세요.
모르면 모른다고 말하세요.

컨텍스트:
{context}

질문: {input}"""
)

# 1) 문서 + LLM을 결합하는 체인
question_answer_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt,
    document_variable_name="context",
)

# 2) Retriever와 합쳐서 최종 RAG 체인 구성
qa_chain = create_retrieval_chain(
    retriever=retriever,
    combine_docs_chain=question_answer_chain,
)

### 4. Demian 내용에 질의하기

In [70]:
query = "데미안에서 싱클레어가 처음으로 '표식(mark)'에 대해 깨닫는 장면이 뭐야?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 싱클레어가 '표식(mark)'에 대해 처음으로 깨닫는 장면은 성경 수업 중에 목사가 가인과 아벨의 이야기를 하면서 가인의 표식에 대해 열정적으로 이야기할 때 발생합니다. 그 순간 싱클레어는 데미안의 시선을 느끼고, 목사의 말에 깊은 관심을 가지게 되며, 목사가 가르치는 내용이 옳지 않다는 것을 깨닫고 대안적인 해석이 가능하다는 것을 느낍니다. 이로 인해 싱클레어와 데미안 사이에 새로운 유대감이 형성됩니다.

[출처 페이지들]
55 DJ:MIAN 
one day in the early morning class when the light was 
still burning in ...
53 DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thr ...
57 DEMIAN 
hoped that the pastor would not ask me any questions, 
he came to my res ...
107 DEMIAN 
I had anticipated. It wu ugly and somewhat wild 
looking, interrogative  ...


In [71]:
query = "How Demian looks like"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 Demian's appearance is described as elegant and at ease, with a face that seems neither fully masculine nor feminine, neither old nor young, but rather timeless and bearing marks of different historical periods. His expression conveys a deep, quiet, almost fanatical yet passionate absorption. At one point, he is compared to a pale, stone mask, suggesting a cold and handsome demeanor with an underlying secret life. Overall, he has an aura of quiet emptiness and a sense of being beyond reach.

[출처 페이지들]
53 DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thr ...
93 DEMIAN 
dent. I could not forget him: And the worda he had said 
in that tavern  ...
89 DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often ...
70 THE THIEF 
eyes were fixed on that pale, stone mask, speJibound, and 
I felt tha ...


#### 위 답변:

Demian's appearance is described as elegant and at ease, with a face that seems neither fully masculine nor feminine, neither old nor young, but rather timeless and bearing marks of different historical periods. His expression conveys a deep, quiet, almost fanatical yet passionate absorption. At one point, he is compared to a pale, stone mask, suggesting a cold and handsome demeanor with an underlying secret life. Overall, he has an aura of quiet emptiness and a sense of being beyond reach.



#### 아래 Naive REG 사용시 답변:

Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet absorption. His face is noted to be different from others, not distinctly masculine or feminine, and it carries a timeless quality, almost as if it bears the marks of different periods of history. The narrator feels that Demian is unimaginarily different, likening him to an animal, a spirit, or an image. At one point, he is described as having a stone-like, age-old appearance, handsome yet cold, with an aura of quiet emptiness. Overall, Demian's appearance evokes a sense of mystery and depth, making him stand out from those around him.답변: Demian is described as having an expression that is elegant and at ease, with a glance that shows deep, quiet absorption. His face is noted to be different from others, not distinctly masculine or feminine, and it carries a timeless quality, almost as if it bears the marks of different periods of history. The narrator feels that Demian is unimaginarily different, likening him to an animal, a spirit, or an image. At one point, he is described as having a stone-like, age-old appearance, handsome yet cold, with an aura of quiet emptiness. Overall, Demian's appearance evokes a sense of mystery and depth, making him stand out from those around him.

In [85]:
query = "데미안은 어떻게 생겼는지?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 주어진 컨텍스트에서는 데미안의 외모에 대한 구체적인 설명이 없습니다. 그러나 데미안의 어머니인 에바에 대한 묘사가 있으며, 그녀는 "키가 크고 거의 남성적인 모습"을 가지고 있다고 언급되어 있습니다. 데미안이 그녀를 닮았다고도 하니, 데미안도 비슷한 외모를 가졌을 가능성이 있습니다. 하지만 데미안의 외모에 대한 직접적인 정보는 제공되지 않았습니다.

[출처 페이지들]
175 DEMIAN 
How the world had changed I I had been summoning 
all my streng!h to con ...
142 VII 
Eva 
One time during the holidays I visited the house where 
-years before, ...
19 DEMIAN 
father pronounced the bleuing, and when he ended 
11God keep us all • .  ...
57 DEMIAN 
hoped that the pastor would not ask me any questions, 
he came to my res ...


In [86]:
query = "데미안이 Demian's appearance is described as elegant and at ease 인지 어떻게 알아?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 데미안의 외모가 "elegant and at ease"로 묘사된 것은 그가 뒤쪽에 서서 말의 머리를 바라보는 모습에서 나타납니다. 그의 표정은 깊고 조용하며, 거의 광신적이면서도 열정적인 몰입을 보여주기 때문입니다. 이러한 모습은 그가 우아하고 편안한 인상을 준다는 것을 암시합니다.

[출처 페이지들]
53 DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thr ...
107 DEMIAN 
I had anticipated. It wu ugly and somewhat wild 
looking, interrogative  ...
145 DEMIAN 
.. I don't suppose it is any better with you in Japan. 
The people who d ...
93 DEMIAN 
dent. I could not forget him: And the worda he had said 
in that tavern  ...


In [76]:
query = "데미안과 싱클레어의 관계는 처음에 어떻게 시작되고, 시간이 지나면서 어떻게 변해?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 데미안과 싱클레어의 관계는 처음에 싱클레어가 자신의 비밀과 두려움으로 고통받고 있을 때, 데미안이 그에게 중요한 존재로 등장하면서 시작됩니다. 데미안은 싱클레어에게 새로운 시각과 이해를 제공하며, 그가 겪고 있는 '다른 세계'에 대한 인식을 확장시킵니다. 

시간이 지나면서, 싱클레어는 데미안과의 관계를 통해 자신의 내면을 탐구하고 성장하게 됩니다. 그러나 그 과정에서 싱클레어는 사랑과 존경을 바탕으로 한 관계가 자연스럽게 그를 사랑하는 사람들과의 거리감을 초래할 수 있다는 것을 깨닫게 됩니다. 결국, 그들은 서로에게 중요한 존재이지만, 싱클레어는 자신의 길을 가기 위해 데미안과의 관계에서 벗어나야 하는 상황에 직면하게 됩니다. 이러한 변화는 싱클레어가 자신의 정체성을 찾고 성장하는 과정에서 불가피한 일로 나타납니다.

[출처 페이지들]
5 DEMIAN 
confessions, forgiveness and good resolutions, love and 
reverence, wisd ...
43 DEMIAN 
the nature of hope. I was no longer alone I And now for 
the first time  ...
51 DEMIAN .... 
cling desperately their whole life through to the irrevo­
cable pas ...
133 DEMIAN 
from them. I was sorry and suffered many galling hours 
during my visits ...


In [77]:
query = "데미안은 싱클레어에게 어떤 역할(멘토, 친구, 또는 다른 무엇)을 하나?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 데미안은 싱클레어에게 멘토이자 친구의 역할을 합니다. 그는 싱클레어가 자신의 내면을 탐구하고, 고독을 극복하며, 새로운 가능성을 인식하도록 돕습니다. 데미안은 싱클레어에게 사랑과 영혼의 중요성을 가르치고, 그가 자신의 정체성을 찾는 과정에서 중요한 영향을 미치는 인물입니다.

[출처 페이지들]
157 DEMIAN 
like a son and brother-and a lover too. When I closed 
the door behind m ...
115 DEMIAN 
young nine months I You can see how many of thein are 
fish or sheep, wo ...
3 DEMIAN 
is important, eternal, sacred; and why every man while 
he lives and ful ...
133 DEMIAN 
from them. I was sorry and suffered many galling hours 
during my visits ...


In [80]:
query = "데미안의 어머니 에바 부인과 싱클레어 사이의 관계를 데미안은 어떻게 매개하고 있어?"
result = qa_chain.invoke({"input": query})

print("답변:\n", result["answer"])
print("\n[출처 페이지들]")
for doc in result["context"]:
    print(doc.metadata.get("page", "?"), doc.page_content[:80], "...")

답변:
 데미안은 싱클레어와 그의 어머니 에바 부인 사이의 관계를 매우 특별하게 매개하고 있습니다. 그는 싱클레어가 에바 부인을 'Frau Eva'라고 부르며 그녀를 사랑하고 있다는 것을 인식하고, 이를 통해 싱클레어의 감정을 이해합니다. 데미안은 싱클레어가 에바 부인에게 특별한 애정을 가지고 있다는 것을 알고 있으며, 그 애정이 싱클레어와 에바 부인 사이의 깊은 연결을 나타낸다고 생각합니다. 또한, 데미안은 에바 부인이 싱클레어에게 특별한 의미를 지니고 있다는 것을 알고 있으며, 이를 통해 두 사람의 관계가 더욱 깊어질 수 있도록 돕고 있습니다.

[출처 페이지들]
142 VII 
Eva 
One time during the holidays I visited the house where 
-years before, ...
156 match; he's as agile as a kitten, and just as full of tricks, 
of course. But he ...
175 DEMIAN 
How the world had changed I I had been summoning 
all my streng!h to con ...
61 no reply, and though I was profoundly curious, I could 
not repeat the question. ...


In [ ]:
# 사용형식
# result = qa_chain.invoke({"input": "여기에 질문"})
# print(result["answer"])          # 최종 답변
# print(result["context"])         # 사용된 문서들

### 여기까지 pdf 문서 테스트 종료

# 아래는 기존 Naive REG 코드

#### CSVLoader

CSV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [72]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("titanic.csv")

data = loader.load()

In [73]:
data[:3]

[Document(metadata={'source': 'titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': 'titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': 'titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [74]:
from langchain_community.document_loaders import WebBaseLoader

In [75]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

KeyboardInterrupt: 

In [ ]:
#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [ ]:
from langchain_text_splitters import CharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [ ]:
with open("state_of_the_union.txt") as f:
    text = f.read()

In [ ]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [ ]:
print(chunks[0])

각 chunk의 길이를 확인해보겠습니다,

In [ ]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [ ]:
!pip install tiktoken

In [ ]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [ ]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
Google의 Gemini 계열 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

공식 문서는 아래 링크에서 확인할 수 있습니다.
https://ai.google.dev/docs/embeddings_guide?hl=ko

In [ ]:
# import google.generativeai as genai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [ ]:
# genai.configure(api_key=YOUR_API_KEY)

In [ ]:
# for model in genai.list_models():
#     if "embedContent" in model.supported_generation_methods:
#         print(model.name)

두 개의 임베딩 모델을 사용할 수 있습니다  

https://github.com/google/generative-ai-docs/blob/main/examples/gemini/python/langchain/Gemini_LangChain_QA_Chroma_WebLoad.ipynb

In [ ]:
# from langchain_google_genai import GoogleGenerativeAIEmbeddings
# embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

Gemini embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [ ]:
# OpenAI embedding 사용
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

# OpenAI 임베딩 모델 생성 (추천: text-embedding-3-small / large)
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"  # 또는 "text-embedding-3-large"
)

In [ ]:
text = "임베딩 테스트 문장입니다."
vector = embedding_model.embed_query(text)
print(len(vector), vector[:5])  # 벡터 길이, 일부만 출력


In [ ]:
#!curl ipinfo.io

In [ ]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 google의 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [ ]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [ ]:
print(embeddings[1])

In [ ]:
len(embeddings[1])

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [ ]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
query = ["this is red fruit"]

In [ ]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [ ]:
!pip install chromadb

In [ ]:
#!pip install langchain-chroma

In [ ]:
#from langchain_community.vectorstores import Chroma

In [ ]:
from langchain_community.vectorstores import Chroma

In [ ]:
#!pip install --upgrade opentelemetry-api
#!pip install --upgrade opentelemetry-sdk

In [ ]:
#from langchain_chroma import Chroma

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [ ]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [ ]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [ ]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [ ]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [ ]:
print(docs[0].page_content)

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [ ]:
!pip install -q langchain-classic


In [ ]:
from langchain_classic.chains import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [ ]:
# from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# llm = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

In [ ]:
from langchain_core.callbacks import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",  # 또는 "gpt-4o"
    temperature=0.0,
    streaming=True,  # 스트리밍 활성화
    callbacks=[StreamingStdOutCallbackHandler()]  # 실시간 출력
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [ ]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [ ]:
query = "how demian looks like"
result = qa(query)

마크다운 형식으로 출력해봅니다

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [ ]:
from langchain_openai import ChatOpenAI

# llm2 = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)
llm2 = ChatOpenAI(
    model="gpt-4o-mini",  # 또는 "gpt-4o"
    temperature=0.0
)

request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


In [ ]:
# 추가 테스트

query = "How Demian looks like"
result = qa(query)

# 1. 답변 출력
print("답변:", result['result'])
print("-" * 50)

# 2. 출처 문서 확인 (핵심!)
print("출처 문서들:")
for i, doc in enumerate(result['source_documents'], 1):
    print(f"\n📄 문서 {i}:")
    print(f"내용: {doc.page_content[:200]}...")  # 처음 200자
    print(f"메타데이터: {doc.metadata}")
    print("-" * 30)


### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
다음은 Lagnchain으로 구현된 Question-Answer RAG 완성 예제입니다  


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
YOUR_API_KEY = ''

필요한 라이브러리를 모두 다운받습니다  

In [ ]:
!pip install -q langchain langchain-google-genai chromadb pypdf sentence_transformers tiktoken

In [ ]:
!pip install U -q langchain-community langchain-core

In [ ]:
import os
os.environ['GOOGLE_API_KEY'] = YOUR_API_KEY

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import PyPDFLoader
from langchain.vectorstores import Chroma

Text splitter 사용을 위한 준비입니다

In [ ]:
import tiktoken

tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(text)

    return len(tokens)

### Step 1 Document loader

In [ ]:
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()



### Step 2 Text splitters

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=5000, chunk_overlap=50, length_function = tiktoken_len)
texts = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
docsearch = Chroma.from_documents(texts, embedding_model)

### Step 4 Retrievers

In [ ]:
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
# QA

llm_gemini = ChatGoogleGenerativeAI(model = "gemini-1.5-flash", temperature=0.0)

In [ ]:
qa = RetrievalQA.from_chain_type(llm_gemini, chain_type="stuff",
                                 retriever=docsearch.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10 }
                                 ))

### Question Answering

In [ ]:
query = "how demian looks like"
result = qa(query)

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))